# MCP tool-permission comparison

Run the real repository engine on fixed, labeled synthetic fixtures in your own notebook session. This is a guided engine demonstration, **not the complete local web application** and not a production safety assessment.

1. Sign in to Colab with your Google account. Select its **free CPU runtime**; no subscription, GPU, compute-unit purchase, payment setup or API key is required by this notebook.
2. Run the setup cell. It resolves the selected Git branch once, checks out that exact SHA and creates a separate temporary Python environment.
3. Choose a scenario and run the next cell. Change it and rerun, or run the side-by-side cell. Expand the JSON evidence and inspect hashes, outcomes and limitations.

The notebook uses its normal inline interface. It starts **no web server, tunnel, background worker, scheduled job or cloud resource** and mounts no Google Drive. Colab can limit or terminate free sessions; no availability guarantee is made. Uploaded user data, credentials and PHI are not requested. Cells contain no saved outputs. Editing this notebook changes the demonstrated scope.

The setup downloads public source and packages. Your local product has separate prerequisites and a broader UI: see the repository [README](https://github.com/poojakira/mcp-agent-security-gateway/tree/codex/local-workbench-20261116). Local execution of these cells is recorded separately from an actual Colab-hosted execution; neither an opening link nor a local pass proves that every future Colab runtime works.


In [ ]:
# Run in Colab's free CPU runtime. No API key, Drive mount, GPU or paid service.
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

from IPython.display import JSON, display

GIT = shutil.which("git")
if GIT is None:
    raise RuntimeError("Git is required for source verification")
GIT_URL = "https://github.com/poojakira/mcp-agent-security-gateway.git"
SOURCE_REF = "refs/heads/codex/local-workbench-20261116"
resolved = subprocess.check_output([GIT, "ls-remote", GIT_URL, SOURCE_REF], text=True).split()
if len(resolved) != 2 or len(resolved[0]) != 40:
    raise RuntimeError("Reviewed branch could not be resolved; no engine run was attempted")
SOURCE_COMMIT = resolved[0]
checkout_directory = tempfile.TemporaryDirectory(prefix="mcp-agent-security-gateway-")
ROOT = Path(checkout_directory.name)
subprocess.run([GIT, "init", "-q", str(ROOT)], check=True)
subprocess.run([GIT, "-C", str(ROOT), "remote", "add", "origin", GIT_URL], check=True)
subprocess.run([GIT, "-C", str(ROOT), "fetch", "--depth=1", "origin", SOURCE_COMMIT], check=True)
subprocess.run([GIT, "-C", str(ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
assert (
    subprocess.check_output([GIT, "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    == SOURCE_COMMIT
)
# Keep dependencies in a separate temporary environment, outside the clean checkout.
environment_directory = tempfile.TemporaryDirectory(prefix="synthetic-demo-environment-")
ENV_ROOT = Path(environment_directory.name)
subprocess.run([sys.executable, "-m", "venv", str(ENV_ROOT)], check=True)
DEMO_PYTHON = ENV_ROOT / "bin" / "python"
subprocess.run(
    [
        str(DEMO_PYTHON),
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-cache-dir",
        "-r",
        str(ROOT / "demo" / "requirements-notebook.txt"),
    ],
    check=True,
)
print("Resolved source:", SOURCE_COMMIT)
print("Guided dependencies installed. Full-app and hosted-runtime verification are separate.")

In [ ]:
# Select a scenario in the Colab form, then run this cell. Repeat with another scenario.
scenario = "read"  # @param ["read", "deny", "allow"]

if scenario not in ["read", "deny", "allow"]:
    raise ValueError("Choose a documented synthetic scenario")
completed = subprocess.run(
    [str(DEMO_PYTHON), "-I", str(ROOT / "demo" / "notebook_runner.py"), scenario],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
    timeout=60,
)
result = json.loads(completed.stdout)
print(result["report"]["evidence"]["summary"])
print("Run source:", result["report"]["source"])
print("Evidence SHA-256:", result["report_sha256"])
display(JSON(result, expanded=False))

In [ ]:
# Run every fixed scenario for an explicit side-by-side engineering comparison.
# Results are recomputed by the actual engine, not loaded from remembered output.
all_results = []
for item in ["read", "deny", "allow"]:
    completed = subprocess.run(
        [str(DEMO_PYTHON), "-I", str(ROOT / "demo" / "notebook_runner.py"), item],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
        timeout=60,
    )
    value = json.loads(completed.stdout)
    all_results.append(value)
    print(item, "→", value["report"]["evidence"]["summary"])
display(JSON(all_results, expanded=False))